![image.png](https://i.imgur.com/4fN73lZ.png)

This notebook has been inspired from [REINFORCEMENT LEARNING (DQN) TUTORIAL](https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) and [reinforcement_q_learning](https://colab.research.google.com/github/pytorch/tutorials/blob/gh-pages/_downloads/reinforcement_q_learning.ipynb#scrollTo=ChfTgUJGcvEA) by Paszke & Towers.

### Setup

We standardize on **Gymnasium** plus `imageio` (GIF rendering) and install with **uv**. `torch` is the deep-learning backend.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
# (torch ships with Colab; locally `uv sync` installs it.)
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control]" imageio matplotlib torch

# Deep Q-Learning

> **Exercise:** This is the student version. Complete the three tasks (`TASK 1` … `TASK 3`) in the helper-functions cell: the greedy action in `select_action`, and the `Q(s, a)` prediction plus the TD target in `optimize_model`. Unfinished tasks raise `NotImplementedError`. Hints give the *formula*, not the code. A fully worked version is in `Day-2_DQN_Solution.ipynb`.

In this notebook, we will implement Deep Q-Learning Reinforcement learning algorithm for Cart Pole Environment.

## Cartpole

As the image below shows, the goal of the agent is to balance a vertical pole on a moving cart. This position is unstable, which is what makes the task difficult.

![Cartpole](https://gymnasium.farama.org/_images/cart_pole.gif)

**Reward.** The agent receives a reward of **+1 for every timestep the pole stays upright**. The episode ends the moment the pole falls over or the cart leaves the track, so the only way to earn more reward is to keep the pole balanced for more timesteps. **An episode's total reward therefore equals the number of timesteps it lasted.**

The episode ends as soon as any of these happens:
- the pole tilts more than ±12° from vertical, or
- the cart moves more than ±2.4 from the centre (it reaches the edge of the track), or
- the episode reaches the **500-timestep limit** (the time cap for CartPole-v1; the older v0 capped at 200).

So the **maximum possible score is 500** (the pole was balanced for all 500 timesteps). CartPole-v1 is usually considered "solved" when the average score over 100 episodes reaches about 475.

The state is 4-dimensional: cart position, cart velocity, pole angle, and pole angular velocity.

You can read more about the CartPole environment [here](https://gymnasium.farama.org/environments/classic_control/cart_pole/).

## Deep Q-Learning

The main idea behind Q-learning is that if we had a function
$Q^*: State \times Action \rightarrow \mathbb{R}$, that could tell
us what our return would be, if we were to take an action in a given
state, then we could easily construct a policy that maximizes our
rewards:

\begin{align}\pi^*(s) = \arg\!\max_a \ Q^*(s, a)\end{align}

But this is not scalable. Must compute $Q(s,a)$ for every state-action pair. If state is e.g. current game state pixels, computationally infeasible to compute for entire state space! But, since neural networks are universal function
approximators, we can simply create one and train it to resemble
$Q^*$.

For our training update rule, we'll use a fact that every $Q$
function for some policy obeys the Bellman equation:

\begin{align}Q^{\pi}(s, a) = r + \gamma Q^{\pi}(s', \pi(s'))\end{align}

The difference between the two sides of the equality is known as the
temporal difference error, $\delta$:

\begin{align}\delta = Q(s, a) - (r + \gamma \max_a Q(s', a))\end{align}

To minimise this error, we will use the `Huber
loss <https://en.wikipedia.org/wiki/Huber_loss>`__. The Huber loss acts
like the mean squared error when the error is small, but like the mean
absolute error when the error is large - this makes it more robust to
outliers when the estimates of $Q$ are very noisy. We calculate
this over a batch of transitions, $B$, sampled from the replay
memory:

\begin{align}\mathcal{L} = \frac{1}{|B|}\sum_{(s, a, s', r) \ \in \ B} \mathcal{L}(\delta)\end{align}

\begin{align}\text{where} \quad \mathcal{L}(\delta) = \begin{cases}
     \frac{1}{2}{\delta^2}  & \text{for } |\delta| \le 1, \\
     |\delta| - \frac{1}{2} & \text{otherwise.}
   \end{cases}\end{align}



In [ ]:
import gymnasium as gym
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import namedtuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# if gpu is to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Create the environment
env = gym.make("CartPole-v1", render_mode="rgb_array")

### Experience Replay

Learning from batches of consecutive samples is problematic as the sample are correlated and it can create a bad feedback loop if one action is dominated in the samples.

We can address these problems using an experience replay memory. It maintains a record for all the transitions experienced. The agent is then trained by sampling random minibatches from the replay memory.

In [ ]:
# Named tuple representing a single transition in our environment
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward', 'done'))

# Cyclic buffer of bounded size that holds and samples the transitions observed recently
class ReplayMemory(object):
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []
        self.position = 0

    def push(self, *args):
        """Saves a transition."""
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = Transition(*args)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

### Q-Network

In [ ]:
# A neural network approximater for Q-Value.
# It takes the state as input and predicts the Q-value for all actions at that state
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

### Exploration vs Exploitation

Notice that Q-learning only learns about the states and actions it visits. What if an optimal state remains unvisited due to not being explored. The agent should sometimes pick suboptimal actions in order to visit new states and actions. <br>

A simple strategy is to use an $\epsilon$-greedy policy. According to this policy, the agent takes a random action with epsilon probability. The value of epsilon is high at the start of training and low towards the end. So, the agent explores more at the start and then exploit the learned policy more at the end.

### Hyperparameters

In [ ]:
# Hyperparameters
total_episodes = 600          # Total training episodes
max_steps = 500               # Max steps per episode (CartPole-v1 truncates at 500)
learning_rate = 1e-3          # Optimizer learning rate
gamma = 0.99                  # Discount factor
batch_size = 128              # Minibatch size sampled from replay memory
target_update = 1000          # Sync target network every C environment steps
memory_capacity = 10000       # Replay buffer size

# Exploration schedule (epsilon-greedy, annealed exponentially)
max_epsilon = 1.0
min_epsilon = 0.01
decay_rate = 0.01

### Training

In [ ]:
print('observation space:', env.observation_space)
print('action space:', env.action_space)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n
print('state size:', state_size, '| action size:', action_size)

# Two networks: policy_net is trained every step; target_net provides stable targets
policy_net = DQN(state_size, action_size).to(device)
target_net = DQN(state_size, action_size).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
memory = ReplayMemory(memory_capacity)


def to_tensor(obs):
    """Convert a single env observation into a (1, state_size) float tensor."""
    return torch.tensor(np.asarray(obs), dtype=torch.float32, device=device).unsqueeze(0)

In [ ]:
def select_action(state, epsilon):
    """Epsilon-greedy over the policy network. Returns a (1, 1) long tensor."""
    if random.random() < epsilon:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)
    with torch.no_grad():
        # TASK 1: return the GREEDY action as a (1, 1) long tensor.
        # HINT: policy_net(state) has shape (1, n_actions). Take the argmax over dim 1
        #       and reshape to (1, 1) (e.g. with .view(1, 1)).
        raise NotImplementedError("TASK 1: implement the greedy action")


def optimize_model():
    """One gradient step of DQN on a random minibatch from replay memory."""
    if len(memory) < batch_size:
        return
    transitions = memory.sample(batch_size)
    batch = Transition(*zip(*transitions))

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    next_state_batch = torch.cat(batch.next_state)
    reward_batch = torch.cat(batch.reward)
    done_batch = torch.cat(batch.done)

    # TASK 2: compute Q(s, a) for the actions ACTUALLY taken in this batch.
    # HINT: policy_net(state_batch) has shape (batch, n_actions), but you only want the Q-value
    #       of the action actually taken in each transition (those indices are in action_batch,
    #       shape (batch, 1)). torch.gather along dim 1 selects one entry per row. Result: (batch, 1).
    state_action_values = None

    # max_a' Q_target(s', a') from the FROZEN target network (provided)
    with torch.no_grad():
        next_state_values = target_net(next_state_batch).max(1).values

    # TASK 3: build the TD target  r + gamma * max_a' Q_target(s', a').
    # HINT: use reward_batch, gamma and next_state_values. Multiply the bootstrap term by
    #       (1 - done_batch) so it becomes 0 at terminal transitions (no next state).
    target_q_values = None

    if state_action_values is None or target_q_values is None:
        raise NotImplementedError("Complete TASK 2 and TASK 3")

    # Huber loss between prediction and target, then a single clipped optimizer step
    loss = F.smooth_l1_loss(state_action_values, target_q_values.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)  # stabilises training
    optimizer.step()
    return loss.item()

In [ ]:
rewards = []
epsilon = max_epsilon
global_step = 0

for episode in range(1, total_episodes + 1):
    state, _ = env.reset()
    state = to_tensor(state)
    total_rewards = 0

    for t in range(max_steps):
        action = select_action(state, epsilon)
        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        total_rewards += reward

        # Store the transition (everything as tensors so we can batch later)
        reward_t = torch.tensor([reward], dtype=torch.float32, device=device)
        done_t = torch.tensor([float(done)], device=device)
        next_state = to_tensor(next_obs)
        memory.push(state, action, next_state, reward_t, done_t)

        state = next_state
        optimize_model()          # learn every step from a random minibatch
        global_step += 1

        # Periodically copy policy weights into the target network (every C steps)
        if global_step % target_update == 0:
            target_net.load_state_dict(policy_net.state_dict())
        if done:
            break

    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    rewards.append(total_rewards)
    if episode % 20 == 0:
        print(f"Episode {episode:4d} | avg reward (last 20): {np.mean(rewards[-20:]):6.1f} | epsilon: {epsilon:.3f}")

In [ ]:
def moving_average(x, window):
    x = np.asarray(x, dtype=float)
    return x if len(x) < window else np.convolve(x, np.ones(window) / window, mode="valid")

plt.figure(figsize=(12, 4))
plt.plot(rewards, alpha=0.3, label="per-episode reward")
plt.plot(np.arange(len(rewards) - len(moving_average(rewards, 20)), len(rewards)),
         moving_average(rewards, 20), label="moving average (20)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Visualization

In [ ]:
# Visualization helpers (Gymnasium-native, no dependency on the old `gym` package)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

def record_gif(env, name, max_steps=500, fps=30):
    """Roll out the greedy policy (policy_net) and save the episode as a GIF."""
    frames = []
    state, _ = env.reset()
    for _ in range(max_steps):
        frames.append(env.render())
        with torch.no_grad():
            action = policy_net(to_tensor(state)).max(1).indices.item()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    env.close()
    path = f"video/{name}.gif"
    imageio.mimsave(path, frames, fps=fps, loop=0)
    return path

def show_gif(name):
    display(Image(filename=f"video/{name}.gif"))

In [ ]:
eval_env = gym.make("CartPole-v1", render_mode="rgb_array")
record_gif(eval_env, "CartPole-v1")
show_gif("CartPole-v1")